# German Macro Regime Dashboard
**Author:** Shardul Pundir | Economics Graduate · CFA Level I  
**Updated:** April 2026  

This notebook tracks Germany's economic cycle in real time using five data streams:
IFO Business Climate, ZEW Economic Sentiment, ECB rates & inflation, and DAX sector performance.

**How to use:**
1. Set your FRED API key in Cell 2 (free — takes 2 minutes to register)
2. Run all cells (`Cell → Run All`)
3. Review the regime classification and sector playbook at the bottom


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────────────
# Get your free FRED API key at: https://fred.stlouisfed.org/docs/api/api_key.html
# Registration takes 2 minutes. The key unlocks IFO and ZEW historical data.

FRED_API_KEY = "YOUR_FRED_API_KEY_HERE"   # ← Replace this
START_DATE   = "2019-01-01"               # Historical start date

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date

print("✓ Libraries loaded")
print(f"✓ Analysis period: {START_DATE} → {date.today()}")


## 1. ECB Rates & Inflation
*Source: ECB Statistical Data Warehouse (SDW) — free, no API key required*

The ECB deposit rate is the key policy lever for all EUR assets.  
The 2s10s Bund spread tells you where the market thinks rates are going.


In [ ]:
from ecb_macro import fetch_all_macro, get_yield_curve_shape

print("Fetching ECB rates and inflation from SDW API...")
ecb_df = fetch_all_macro(start_date=START_DATE)
ecb_monthly = ecb_df.resample("MS").last()

print(f"\n{'='*50}")
print("LATEST ECB DATA")
print(f"{'='*50}")
latest_ecb = ecb_df.dropna(how='all').iloc[-1]
print(f"  ECB Deposit Rate : {latest_ecb.get('ecb_deposit_rate', 'N/A'):.2f}%")
print(f"  Bund 2Y          : {latest_ecb.get('bund_2y', 'N/A'):.3f}%")
print(f"  Bund 5Y          : {latest_ecb.get('bund_5y', 'N/A'):.3f}%")
print(f"  Bund 10Y         : {latest_ecb.get('bund_10y', 'N/A'):.3f}%")
print(f"  EA HICP (YoY)    : {latest_ecb.get('hicp_ea', 'N/A'):.1f}%")
print(f"  German HICP      : {latest_ecb.get('hicp_de', 'N/A'):.1f}%")

curve = get_yield_curve_shape(ecb_df)
print(f"\n  2s10s Spread     : {curve['spread_2s10s']:+.3f}% → {curve['curve_regime']}")


In [ ]:
# ── Chart 1: Bund Yield Curve + ECB Rate ────────────────────────────────
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    "Bund Yield Curve vs ECB Deposit Rate",
    "Euro Area & German HICP (YoY %)"
], vertical_spacing=0.12)

for series, color, name in [
    ("bund_2y",  "#3b82f6", "2Y Bund"),
    ("bund_5y",  "#60a5fa", "5Y Bund"),
    ("bund_10y", "#1d4ed8", "10Y Bund"),
    ("ecb_deposit_rate", "#ef4444", "ECB Deposit Rate"),
]:
    s = ecb_df[series].dropna()
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, name=name,
        line=dict(color=color, width=2 if "ecb" in series else 1.5)
    ), row=1, col=1)

for series, color, name in [
    ("hicp_ea", "#8b5cf6", "EA HICP"),
    ("hicp_de", "#a78bfa", "German HICP"),
]:
    s = ecb_monthly[series].dropna()
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, name=name,
        line=dict(color=color, width=2)
    ), row=2, col=1)

# ECB 2% target line
fig.add_hline(y=2.0, line_dash="dot", line_color="gray",
              annotation_text="ECB 2% target", row=2, col=1)

fig.update_layout(
    title="ECB Macro Environment — Germany",
    height=650, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig.update_yaxes(title_text="Rate (%)", row=1, col=1)
fig.update_yaxes(title_text="YoY (%)", row=2, col=1)
fig.show()


## 2. IFO & ZEW: German Confidence Indicators
*Source: FRED API (free key required) — ifo Institute & ZEW Institute data*

**IFO:** Monthly survey of ~9,000 German firms. The most important leading indicator for Germany.  
**ZEW:** Monthly survey of ~350 financial experts on 6-month economic expectations.

**How to read:**
- IFO Expectations > Current = firms see better times ahead (bullish signal)
- ZEW recovering from negative = inflection point (early recovery signal)


In [ ]:
from ifo_data import fetch_ifo_from_fred, compute_ifo_signals, get_ifo_regime_score
from zew_data  import fetch_zew_from_fred, compute_zew_signals, get_zew_regime_score

if FRED_API_KEY == "YOUR_FRED_API_KEY_HERE":
    print("⚠️  FRED API key not set. Using illustrative sample data.")
    print("   Register free at: https://fred.stlouisfed.org/docs/api/api_key.html")
    
    # Sample data to demonstrate the dashboard without API key
    dates = pd.date_range("2022-01-01", periods=28, freq="MS")
    ifo_df = pd.DataFrame({
        "ifo_climate"      : [98.9, 98.4, 97.3, 93.0, 92.1, 88.6, 84.3, 84.4, 84.4,
                               86.3, 86.5, 88.4, 88.5, 89.1, 88.2, 87.3, 85.7, 85.2,
                               87.5, 89.0, 90.1, 90.5, 91.2, 93.0, 94.1, 95.3, 95.8, 96.1],
        "ifo_current"      : [96.6, 97.1, 97.2, 96.5, 99.5, 98.4, 97.7, 97.6, 96.0,
                               95.0, 94.5, 93.5, 93.0, 92.6, 92.0, 91.5, 88.3, 86.2,
                               88.1, 89.5, 90.1, 91.0, 91.5, 92.0, 92.5, 93.0, 93.5, 93.8],
        "ifo_expectations" : [101.2, 99.7, 97.4, 89.6, 85.0, 79.0, 71.4, 71.5, 73.1,
                               77.8, 78.6, 83.5, 84.1, 85.7, 84.5, 83.2, 83.2, 84.3,
                               87.0, 88.6, 90.1, 90.0, 91.0, 94.0, 95.7, 97.6, 98.1, 98.4],
    }, index=dates)
    
    zew_df = pd.DataFrame({
        "zew_expectations" : [54.3, 54.3, 35.9, -39.3, -41.0, -28.0, -55.3, -36.5, -61.9,
                               -59.2, -36.7, -23.3, 16.9, 28.1, 44.3, 35.8, 31.7, 19.2,
                               42.0, 47.5, 51.1, 26.0, 3.6, -14.0, -26.0, 2.0, 11.3, 15.0],
        "zew_current"      : [54.5, 50.1, 54.2, -30.8, -27.4, -40.7, -46.9, -51.6, -71.4,
                               -72.2, -64.5, -55.0, -45.0, -35.0, -22.5, -14.0, -8.5, -3.2,
                               4.0, 10.2, 17.5, 18.0, 12.0, 5.0, -5.0, -8.0, 11.0, 14.5],
    }, index=dates)
else:
    print("Fetching IFO data from FRED...")
    ifo_df = fetch_ifo_from_fred(FRED_API_KEY, start_date=START_DATE)
    
    print("Fetching ZEW data from FRED...")
    zew_df = fetch_zew_from_fred(FRED_API_KEY, start_date=START_DATE)

ifo_df = compute_ifo_signals(ifo_df)
zew_df = compute_zew_signals(zew_df)

ifo_score = get_ifo_regime_score(ifo_df)
zew_score = get_zew_regime_score(zew_df)

print(f"\n{'='*50}")
print("CONFIDENCE INDICATORS — LATEST")
print(f"{'='*50}")
print(f"  IFO Climate      : {ifo_score['ifo_climate']}")
print(f"  Expectations Gap : {ifo_score['expectations_gap']:+.1f} ({ifo_score['climate_direction']})")
print(f"  IFO Score        : {ifo_score['ifo_score']}/25")
print()
print(f"  ZEW Expectations : {zew_score['zew_expectations']}")
print(f"  ZEW Trend        : {zew_score['zew_trend']}")
print(f"  ZEW Score        : {zew_score['zew_score']}/25")


In [ ]:
# ── Chart 2: IFO & ZEW ────────────────────────────────────────────────────
fig = make_subplots(rows=2, cols=1,
    subplot_titles=["IFO Business Climate — Current vs Expectations",
                    "ZEW Economic Sentiment (6-Month Expectations)"],
    vertical_spacing=0.14)

# IFO
fig.add_trace(go.Scatter(
    x=ifo_df.index, y=ifo_df["ifo_climate"], name="Climate (headline)",
    line=dict(color="#1d4ed8", width=2.5)
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=ifo_df.index, y=ifo_df["ifo_current"], name="Current Situation",
    line=dict(color="#60a5fa", width=1.5, dash="dot")
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=ifo_df.index, y=ifo_df["ifo_expectations"], name="Expectations",
    line=dict(color="#f59e0b", width=1.5, dash="dash")
), row=1, col=1)
fig.add_hline(y=100, line_dash="dot", line_color="gray",
              annotation_text="Long-run avg", row=1, col=1)

# ZEW
fig.add_trace(go.Bar(
    x=zew_df.index, y=zew_df["zew_expectations"], name="ZEW Expectations",
    marker_color=["#ef4444" if v < 0 else "#22c55e"
                  for v in zew_df["zew_expectations"]],
), row=2, col=1)
fig.add_hline(y=0, line_color="black", line_width=1, row=2, col=1)

fig.update_layout(
    title="German Business & Investor Confidence",
    height=620, template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig.show()


## 3. DAX Sector Performance
*Source: Yahoo Finance (yfinance) — live prices, no API key required*

Sector rotation reveals what the market is pricing in.  
Cyclicals (Autos, Banks, Industrials) outperforming = expansion.  
Defensives (Utilities, Healthcare) outperforming = contraction risk.


In [ ]:
from dax_data import fetch_returns, compute_performance, get_sector_regime_alignment, SECTOR_TICKERS

print("Fetching DAX sector data from Yahoo Finance...")
sector_prices = fetch_returns(start_date="2023-01-01", tickers=SECTOR_TICKERS)
perf = compute_performance(sector_prices)
regime_signal = get_sector_regime_alignment(sector_prices)

print(f"\n{'='*50}")
print("DAX SECTOR PERFORMANCE")
print(f"{'='*50}")
print(perf[["price", "return_1m", "return_3m", "return_ytd"]].to_string())
print(f"\n  Market Signal: {regime_signal.get('market_regime_signal', 'N/A')}")
print(f"  Cyclical avg (3m): {regime_signal.get('cyclical_avg_3m', 'N/A')}%")
print(f"  Defensive avg (3m): {regime_signal.get('defensive_avg_3m', 'N/A')}%")


In [ ]:
# ── Chart 3: Sector Performance Heatmap ────────────────────────────────
perf_sorted = perf.sort_values("return_ytd", ascending=True)

# Color: green = positive, red = negative
colors = ["#22c55e" if v > 0 else "#ef4444"
          for v in perf_sorted["return_ytd"].fillna(0)]

fig = go.Figure(go.Bar(
    x=perf_sorted["return_ytd"],
    y=perf_sorted.index,
    orientation="h",
    marker_color=colors,
    text=[f"{v:.1f}%" for v in perf_sorted["return_ytd"].fillna(0)],
    textposition="outside",
))

fig.update_layout(
    title="DAX Sector Performance — Year to Date (%)",
    xaxis_title="Return (%)",
    template="plotly_white",
    height=450,
    showlegend=False,
)
fig.show()

# ── Chart 4: Normalised sector price trends ────────────────────────────
fig2 = go.Figure()
base_date = sector_prices.index[0]
for col in sector_prices.columns:
    s = sector_prices[col].dropna()
    if len(s) > 0:
        normalised = (s / s.iloc[0] - 1) * 100
        fig2.add_trace(go.Scatter(
            x=normalised.index, y=normalised.values,
            name=col, mode="lines", line=dict(width=1.5)
        ))

fig2.add_hline(y=0, line_color="black", line_width=1)
fig2.update_layout(
    title=f"DAX Sector Price Performance (indexed to 0, since {base_date.strftime('%b %Y')})",
    yaxis_title="Cumulative Return (%)",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig2.show()


## 4. Regime Classification
*Combining all signals into a single German economic cycle read*


In [ ]:
from regime_classifier import get_ecb_regime_score, get_equity_regime_score, classify_regime, print_regime_summary

# Score each component
ecb_score    = get_ecb_regime_score(ecb_df)
equity_score = get_equity_regime_score(regime_signal)

# Classify
result = classify_regime(ifo_score, zew_score, ecb_score, equity_score)
print_regime_summary(result)


In [ ]:
# ── Chart 5: Component Score Breakdown ─────────────────────────────────
components = result["component_scores"]
labels  = [k.upper() for k in components.keys()]
scores  = list(components.values())
max_scores = [25] * 4

regime_colors = {
    "Expansion"  : "#22c55e",
    "Slowdown"   : "#f59e0b",
    "Contraction": "#ef4444",
    "Recovery"   : "#3b82f6",
}
bar_color = regime_colors.get(result["regime"], "#6b7280")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=scores, name="Score",
    marker_color=bar_color,
    text=[f"{s}/25" for s in scores],
    textposition="outside",
))
fig.add_trace(go.Bar(
    x=labels, y=[25 - s for s in scores], name="Remaining",
    marker_color="lightgray", opacity=0.5
))

fig.update_layout(
    title=f"German Macro Regime: <b>{result['regime'].upper()}</b> [{result['composite_score']}/100] — Confidence: {result['regime_confidence']}",
    yaxis_title="Score (max 25 each)",
    barmode="stack",
    template="plotly_white",
    height=420,
    showlegend=False,
    annotations=[
        dict(x=0.5, y=-0.15, xref="paper", yref="paper",
             text=f"<i>{result['regime_description'][:120]}...</i>",
             showarrow=False, font=dict(size=11, color="gray"))
    ]
)
fig.show()

# Sector playbook
print("\n📋 SECTOR PLAYBOOK")
print("-" * 40)
pb = result["sector_playbook"]
print(f"  ▲ Overweight : {', '.join(pb['overweight'])}")
print(f"  ▼ Underweight: {', '.join(pb['underweight'])}")
print(f"  🔗 Bond view  : {pb['bond_view']}")


## 5. Weekly Macro Commentary Template

Run this cell each week to generate a structured macro update. Good practice for writing PM-style commentaries — essential skill for AM/WM roles.


In [ ]:
# ── Weekly macro commentary builder ────────────────────────────────────
from datetime import date

commentary = f"""
GERMAN MACRO WEEKLY — {date.today().strftime('%d %B %Y')}
{'='*55}

REGIME: {result['regime'].upper()} ({result['composite_score']}/100) — {result['regime_confidence']} confidence

KEY DATA:
  • IFO Business Climate : {ifo_score['ifo_climate']} (expectations gap: {ifo_score['expectations_gap']:+.1f})
  • ZEW Expectations     : {zew_score['zew_expectations']:.1f} ({zew_score['zew_trend']})
  • ECB Deposit Rate     : {ecb_score.get('ecb_rate', 'N/A')}%
  • Bund 10Y             : {ecb_score.get('curve_spread', 'N/A')} spread (2s10s)
  • EA HICP              : {ecb_score.get('hicp_ea', 'N/A')}% YoY
  • Sector Signal        : {equity_score.get('market_signal', 'N/A')}

ANALYSIS:
  {result['regime_description']}

PLAYBOOK:
  Overweight : {', '.join(result['sector_playbook']['overweight'])}
  Underweight: {', '.join(result['sector_playbook']['underweight'])}
  Duration   : {result['sector_playbook']['bond_view']}

WATCH NEXT:
  [ ] IFO release (last Mon/Tue of month)
  [ ] ZEW release (2nd Tue of month)
  [ ] ECB meeting / policy statement
  [ ] German CPI flash estimate
  [ ] PMI Manufacturing (1st Tue of month)

{'='*55}
"""

print(commentary)

# Save to outputs
with open(f"outputs/macro_commentary_{date.today().isoformat()}.txt", "w") as f:
    f.write(commentary)
print(f"Saved → outputs/macro_commentary_{date.today().isoformat()}.txt")
